# 04. 오분류 분석
담당: 데이터 A · 검증셋에서 모델이 틀린 사례를 모아 원인을 찾고, 다음 실험(정제·증강·모델)을 제안합니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.utils import load_class_map, load_config
from src.visualize import set_korean_font

set_korean_font()
pd.set_option("display.max_columns", 50)

CONFIG = "configs/base.yaml"  # 샘플 데이터로 연습: "configs/sample_yolo.yaml"
cfg = load_config(CONFIG)
d = cfg["data"]

In [ ]:
EXP = "sample_yolo"  # 분석할 실험 이름

from src.interfaces import OUTPUT_DIR
from src.metrics import confusion_pairs, match_predictions
from src.train import val_targets
from src.utils import load_json, predictions_from_json

exp_cfg = load_config(OUTPUT_DIR / EXP / "config.yaml")
paths, targets = val_targets(exp_cfg)
preds = predictions_from_json(load_json(OUTPUT_DIR / EXP / "preds_val.json"))
rows = pd.DataFrame(match_predictions(preds, targets, iou_thr=0.75, conf_thr=0.3))
rows["result"].value_counts()

## 자주 헷갈리는 클래스 쌍
같은 쌍이 반복되면 해상도 증가(D6)나 2단계 분류(M4) 실험의 근거가 됩니다.

In [ ]:
class_map = load_class_map(exp_cfg["data"]["class_map"])
names = [c["name"] for c in class_map]
pd.DataFrame(
    [(names[g], names[p], n) for g, p, n in confusion_pairs(rows.to_dict("records"))],
    columns=["정답", "예측", "횟수"],
)

## 틀린 이미지 보기

In [ ]:
from src.dataset import read_image_rgb
from src.visualize import show_prediction

bad_ids = rows.loc[rows["result"] != "correct", "image_id"].unique()[:5]
for p, t, path in zip(preds, targets, paths):
    if t["image_id"] in bad_ids:
        show_prediction(read_image_rgb(path), p, t["boxes"], t["labels"], title=path.name)

## 클래스별 AP는 "어떤 알약을 못 맞히는가"

In [ ]:
import json

import pandas as pd

from src.interfaces import OUTPUT_DIR
from src.utils import load_class_map, load_config

EVAL = "eval_conf0.001_nms0.5_det4.json"
EXPS = {
    "baseline": "1028_modelA_yolov8s_v0_baseline",
    "v3": "1029_dataB_yolov8s_v3_aihub",
    "v3all": "1030_dataB_yolov8s_v3all_aihub",
}


def class_ap(exp):
    """category_id → AP. 실험마다 자기 class_map으로 label_idx를 바꾼다 (v3all은 번호가 달라서)."""
    ev = json.load(open(OUTPUT_DIR / exp / EVAL, encoding="utf-8"))
    if "classes" not in ev:
        raise ValueError(f"{exp}: classes가 없습니다. metrics.py 수정 후 --class-metrics로 다시 실행하세요.")
    cfg = load_config(OUTPUT_DIR / exp / "config.yaml")
    l2c = {c["label_idx"]: c["category_id"] for c in load_class_map(cfg["data"]["class_map"])}
    return pd.Series({l2c[lbl]: ap for lbl, ap in zip(ev["classes"], ev["map_per_class"])})


names = {c["category_id"]: c["name"] for c in load_class_map("data/class_map_v3all.json")}
df = pd.DataFrame({k: class_ap(v) for k, v in EXPS.items()})
df["v3all-base"] = df["v3all"] - df["baseline"]
df.insert(0, "약 이름", [names.get(i, i) for i in df.index])
df.index.name = "category_id"
df.sort_values("v3all-base").round(3)

,약 이름,baseline,v3_aihub,변화
label_idx,,,,
14,크레스토정 20mg,1.000,0.000,-1.000
12,에빅사정(메만틴염산염)(비매품),1.000,0.000,-1.000
22,스토가정 10mg,1.000,0.000,-1.000
21,트루비타정 60mg/병,1.000,0.000,-1.000
19,종근당글리아티린연질캡슐(콜린알포세레이트),1.000,0.000,-1.000
34,큐시드정 31.5mg/PTP,1.000,0.000,-1.000
30,자누비아정 50mg,1.000,0.000,-1.000
23,노바스크정 5mg,1.000,0.000,-1.000
42,레일라정,1.000,0.000,-1.000


## 정리
- 놓친 알약의 공통점:
- 헷갈린 쌍의 공통점:
- 제안하는 다음 실험: